# **TASK 1**

**Chatbots** are computer programs that are designed to have a conversation with humans, either written or spoken. Chatbots use the technology of NLP (Natural Language Processing) to understand human words and grammar, and through NLP they can even handle grammatical or spelling mistakes. Chatbots can seem to "remember" earlier parts of a conversation, but that's because the previous messages are just being re-sent as context each time, they aren't actually learning or updating themselves the way a model does during training.

**Agents** are computer programs that take in data, reason, and take actions to achieve a goal they're designed for. Agents use different tools to analyze data, and those tools help them decide what action to take next. An LLM is used as its brain. What makes an agent "agentic" comes down to a few traits: autonomy (it decides its own next step instead of following a script), tool use (it can take real actions, not just produce text), multi-step planning (it can chain several actions together toward one goal), and self-correction (it can notice a step didn't work and try something different instead of just continuing).

**Workflows** consist of predefined steps to complete a task. The steps and their order are decided in advance, in code, and don't change based on what happens in between, that's the main thing that separates a workflow from an agent, where the model itself decides what happens next based on what it observes.

###**ReAct Pattern**

The ReAct (Reason + Act) pattern is an agent loop where the model reasons about what it needs to do, takes an action by calling a tool, observes the tool's result, and then decides what to do next. This process continues until the model has enough information to provide the final answer.

In [ ]:
while not finished:

    response = model(user_message)

    if response contains tool_use:
        tool_name = response.tool_name
        tool_input = response.tool_input

        result = execute_tool(tool_name, tool_input)

        add tool_result to conversation_history

    else:
        return final_answer

###**when an agent is overkill**
An agent can be overkill when a task consists of predictable steps that don't require much decision-making, or when the task is simple enough that a single prompt or script can handle it. In cases like these, using an agent adds unnecessary complexity, cost, and unpredictability compared to just writing the steps yourself.

# **TASK 2**

In [7]:
from google.colab import userdata
GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')

In [8]:
#geminie sdk
!pip install -q google-genai

In [10]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

In [11]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello in one sentence."
)

print(response.text)

Hello!


### Define Calculator Tool Schema

In [20]:
calculator_tool = {
    "name": "calculator",
    "description": (
        "Evaluates a basic arithmetic expression and returns the numeric result. "
        "Supports addition, subtraction, multiplication, division, and parentheses."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "A mathematical expression, such as '(3 + 4) * 2'."
            }
        },
        "required": ["expression"]
    }
}

### Define File_read tool Schema

In [19]:
read_file_tool = {
    "name": "read_file",
    "description": (
        "Reads a .txt file at the given path and returns its full contents as a string."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "file_path": {
                "type": "string",
                "description": "Path to the .txt file, such as 'notes.txt'."
            }
        },
        "required": ["file_path"]
    }
}

In [21]:
#Display Schema

tools = [calculator_tool, read_file_tool]

print("Calculator Tool:")
print(calculator_tool)

print("\nRead File Tool:")
print(read_file_tool)

Calculator Tool:
{'name': 'calculator', 'description': 'Evaluates a basic arithmetic expression and returns the numeric result. Supports addition, subtraction, multiplication, division, and parentheses.', 'parameters': {'type': 'object', 'properties': {'expression': {'type': 'string', 'description': "A mathematical expression, such as '(3 + 4) * 2'."}}, 'required': ['expression']}}

Read File Tool:
{'name': 'read_file', 'description': 'Reads a .txt file at the given path and returns its full contents as a string.', 'parameters': {'type': 'object', 'properties': {'file_path': {'type': 'string', 'description': "Path to the .txt file, such as 'notes.txt'."}}, 'required': ['file_path']}}


### Why tool descriptions matter:
Tool descriptions help the model understand what each tool does, when it should be used, and what type of information it needs as input. Clear and specific descriptions improve tool selection and reduce errors such as choosing the wrong tool or generating incorrect arguments. For example, describing the calculator as a tool for arithmetic expressions makes it more likely that the model will call it instead of trying to answer the calculation directly.

### Testing Calculator Tool

In [22]:
from google.genai import types

calculator_function = types.FunctionDeclaration(
    name="calculator",
    description="Evaluates a basic arithmetic expression and returns the numeric result.",
    parameters=types.Schema(
        type="OBJECT",
        properties={
            "expression": types.Schema(
                type="STRING",
                description="A mathematical expression such as '(3 + 4) * 2'."
            )
        },
        required=["expression"]
    )
)

tool = types.Tool(
    function_declarations=[calculator_function]
)

In [23]:
# Asking geminie to use the calculator
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Calculate (25 + 15) * 3.",
    config=types.GenerateContentConfig(
        tools=[tool]
    )
)

print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'expression': '(25 + 15) * 3'
          },
          name='calculator'
        ),
        thought_signature=b'\n\x9c\x02\x01\x11M2\x0f\x95\xdc\xfe\xb3\x1a\xaf\xdb\x9c\x0e2C\xd1\xbd\x8fr\'\x9d`K\x8f!\x19/\xb1oB\xb7\x0fF6:\xfbP\xf2qh\xfa\xa3\xff\xf5+$\xd0-]\x9b6\x97Xz\xf8 x\xf2\x08\xb1K.\xa9w\x98\x94\x18X\xd4(\xe9^R\xb6\xcd\x98"F\xb5+\x9b\xa3\xbeq\xfcl\xb9\xc8\xe4\xe2\xa7\xe2v...'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-2.5-flash' prompt_feedback=None response_id='tHqeaub6BpWZ-8YP-9ORoQ0' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=22,
  prompt_token_count=70,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=70
 

In [24]:
#Inspects Geminie tool call
for candidate in response.candidates:
    for part in candidate.content.parts:
        if part.function_call:
            print("Tool name:", part.function_call.name)
            print("Tool arguments:", part.function_call.args)

Tool name: calculator
Tool arguments: {'expression': '(25 + 15) * 3'}


### Manually Executing the Tool

In [26]:
def calculate(expression):
    return eval(expression)


tool_call = response.candidates[0].content.parts[0].function_call

result = calculate(
    tool_call.args["expression"]
)

print("Tool result:", result)

Tool result: 120


### Returing the tool result to geminie

In [27]:
tool_result = types.Part.from_function_response(
    name="calculator",
    response={"result": result}
)

final_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        "Calculate (25 + 15) * 3.",
        response.candidates[0].content,
        tool_result
    ],
    config=types.GenerateContentConfig(
        tools=[tool]
    )
)

print(final_response.text)

The result is 120.


# **TASK 3**

In [28]:
weather_data = {
    "Lahore": {
        "temperature": 35,
        "condition": "Sunny"
    },
    "Islamabad": {
        "temperature": 29,
        "condition": "Cloudy"
    },
    "Karachi": {
        "temperature": 32,
        "condition": "Sunny"
    }
}

In [29]:
#Function
def get_weather(city):
    if city in weather_data:
        return weather_data[city]

    return {
        "error": f"Weather data not available for {city}"
    }

In [30]:
print(get_weather("Lahore"))
print(get_weather("Islamabad"))

{'temperature': 35, 'condition': 'Sunny'}
{'temperature': 29, 'condition': 'Cloudy'}


### Define the Geminie weather tool

In [31]:
weather_function = types.FunctionDeclaration(
    name="get_weather",
    description=(
        "Returns the weather information for a city, "
        "including its temperature and condition."
    ),
    parameters=types.Schema(
        type="OBJECT",
        properties={
            "city": types.Schema(
                type="STRING",
                description="Name of the city to look up."
            )
        },
        required=["city"]
    )
)

### Testing the schema

In [32]:
weather_tool = types.Tool(
    function_declarations=[weather_function]
)

In [33]:
print(weather_function)

description='Returns the weather information for a city, including its temperature and condition.' name='get_weather' parameters=Schema(
  properties={
    'city': Schema(
      description='Name of the city to look up.',
      type=<Type.STRING: 'STRING'>
    )
  },
  required=[
    'city',
  ],
  type=<Type.OBJECT: 'OBJECT'>
) parameters_json_schema=None response=None response_json_schema=None behavior=None


### Build the Agent Tool

In [34]:
def execute_tool(tool_name, tool_input):
    if tool_name == "get_weather":
        return get_weather(tool_input["city"])

    else:
        return {
            "error": f"Unknown tool: {tool_name}"
        }

In [39]:
def run_agent(user_query, max_iterations=5):

    messages = [user_query]
    iteration = 0

    while iteration < max_iterations:

        iteration += 1

        print(f"\n--- Iteration {iteration} ---")

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=messages,
            config=types.GenerateContentConfig(
                tools=[weather_tool]
            )
        )

        # Check if Gemini requested a tool
        function_call = None

        for part in response.candidates[0].content.parts:
            if part.function_call:
                function_call = part.function_call
                break

        # If there is no tool call, return the final answer
        if function_call is None:

            print("Final Answer:")
            print(response.text)

            return response.text

        # Log tool call
        print("Tool:", function_call.name)
        print("Arguments:", function_call.args)

        # Execute the tool
        result = execute_tool(
            function_call.name,
            function_call.args
        )

        # Log observation
        print("Observation:", result)

        # Append Gemini's response to conversation history
        messages.append(response.candidates[0].content)

        # Create tool result
        tool_result = types.Part.from_function_response(
            name=function_call.name,
            response=result
        )

        # Append tool result
        messages.append(tool_result)

    print("Agent stopped: maximum iterations reached.")
    return None

In [40]:
run_agent(
    "Look up the weather in Lahore and Islamabad and tell me which city is warmer."
)


--- Iteration 1 ---
Tool: get_weather
Arguments: {'city': 'Lahore'}
Observation: {'temperature': 35, 'condition': 'Sunny'}

--- Iteration 2 ---
Tool: get_weather
Arguments: {'city': 'Islamabad'}
Observation: {'temperature': 29, 'condition': 'Cloudy'}

--- Iteration 3 ---
Final Answer:
Lahore is warmer than Islamabad. The temperature in Lahore is 35 degrees Celsius and in Islamabad, it is 29 degrees Celsius.


'Lahore is warmer than Islamabad. The temperature in Lahore is 35 degrees Celsius and in Islamabad, it is 29 degrees Celsius.'

# **TASK 4**

### **Conversation Memory vs Working Memory**

**Conversation memory** is the message history shared between the user and the model. It contains the user's request, previous model responses, tool calls, and tool results. In this agent, the `messages` list acts as conversation memory.

**Working memory** is temporary information the agent uses while completing a task. Examples include the current iteration number, tool arguments, and observations returned by tools. Working memory helps the agent decide what to do next but does not necessarily need to become part of the permanent conversation history.

### **Agent Logging**

Logging helps us observe how the agent operates. For each iteration, we record the current step, the selected tool, its arguments, and the observation returned by the tool.

The agent should log observable actions and results rather than exposing hidden chain-of-thought reasoning.

### Simple Logging

In [41]:
def log_step(iteration, tool_name, arguments, observation):
    print(f"\n--- Agent Step {iteration} ---")
    print("Tool:", tool_name)
    print("Arguments:", arguments)
    print("Observation:", observation)

In [42]:
log_step(
    1,
    "get_weather",
    {"city": "Lahore"},
    {"temperature": 35, "condition": "Sunny"}
)


--- Agent Step 1 ---
Tool: get_weather
Arguments: {'city': 'Lahore'}
Observation: {'temperature': 35, 'condition': 'Sunny'}


# **TASK 5**

### Deliberately Breaks the Agent

In [44]:
run_agent(
    "What is the current stock price of Apple?"
)


--- Iteration 1 ---
Final Answer:
I cannot provide real-time stock prices. My capabilities are limited to providing weather information.


'I cannot provide real-time stock prices. My capabilities are limited to providing weather information.'

In [45]:
run_agent(
    "What is the weather in Tokyo?"
)


--- Iteration 1 ---
Tool: get_weather
Arguments: {'city': 'Tokyo'}
Observation: {'error': 'Weather data not available for Tokyo'}

--- Iteration 2 ---
Final Answer:
I'm sorry, I don't have weather information for Tokyo.


"I'm sorry, I don't have weather information for Tokyo."

### Observed Results

- When asked for Apple's current stock price, the agent did not call a tool because no stock-price tool was available. It returned a final response explaining that it could only provide weather information.
- When asked for Tokyo weather, the agent correctly called the `get_weather` tool. The tool returned a structured error because Tokyo was not in the weather dataset. The agent then used the observation to provide a final response explaining that weather information for Tokyo was unavailable.

##**Failure Modes and Mitigation**


1. **Infinite loops:**
       The agent may continue calling tools without reaching a final answer.
    **Mitigation**: Set a max_iterations limit to stop the agent after a fixed number of steps.

2.  **Hallucinated tool calls**: The model may attempt to use a tool that is not available.
    **Mitigation**: Provide only the available tools and use clear tool descriptions so the model understands their capabilities.

3.  **Wrong tool arguments**: The model may provide incorrect or incomplete    parameters to a tool.
    **Mitigation**: Define strict input schemas and validate tool arguments before execution.

4.  **Silent errors**: A tool may fail or return invalid results without the agent clearly handling the problem.
    **Mitigation**: Use structured error responses and explicit error handling so failures are passed back to the agent.

5.  **Wrong tool selection**: The agent may select a tool that does not match the user's request.
    **Mitigation**: Write precise tool descriptions explaining what each tool does and when it should be used.

6.  **Missing capabilities**: The user may request something for which no tool exists.
    **Mitigation**: Clearly inform the user that the required capability is unavailable instead of inventing a result.


**Why Agent Frameworks Exist**

Frameworks such as LangChain, LangGraph, and CrewAI exist because manually building an agent becomes increasingly complex as the number of tools, steps, agents, and state-management requirements grows. The simple agent built in this task handles the basic reasoning loop, tool calling, memory, and error handling by hand, while frameworks provide reusable abstractions for tool management, state and memory, workflows, retries, observability, persistence, and multi-agent coordination. They reduce the amount of infrastructure code developers need to write and maintain when building larger agentic systems.